In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install gradio sentence-transformers faiss-cpu transformers -q

import gradio as gr
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import torch
import warnings
warnings.filterwarnings('ignore')

print("Setup complete.")
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Using device:", device)

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 28.7 MB/s eta 0:00:00
Setup complete.
Using device: cpu


In [2]:
save_path = '/content/drive/MyDrive/rag-complaint-chatbot/vector_store/'

index = faiss.read_index(save_path + 'index.faiss')
print(f"Loaded index with {index.ntotal:,} vectors")

with open(save_path + 'chunks.pkl', 'rb') as f:
    chunked_data = pickle.load(f)
print(f"Loaded {len(chunked_data)} chunks")

embed_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print("Embedding model loaded.")

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

print("Loading LLM...")
model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

if device == 'cuda':
    model = model.to('cuda')

def generate_text(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    if device == 'cuda':
        inputs = {k: v.to('cuda') for k, v in inputs.items()}
    outputs = model.generate(**inputs, max_new_tokens=150)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("LLM loaded successfully.")

Loaded index with 5,291 vectors
Loaded 5291 chunks


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

LLM loaded successfully.


In [3]:
def retrieve(query, k=5):
    query_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    distances, indices = index.search(query_emb, k)
    results = []
    for i, idx in enumerate(indices[0]):
        if idx < len(chunked_data):
            results.append({
                'score': float(distances[0][i]),
                'product': chunked_data[idx]['product_category'],
                'text': chunked_data[idx]['chunk_text']
            })
    return results

def rag_answer(question, k=5):
    retrieved = retrieve(question, k)
    if not retrieved:
        return "No relevant information found.", []

    context = "\n\n".join([f"Excerpt {i+1}: {r['text']}" for i, r in enumerate(retrieved)])

    prompt = f"""You are a financial analyst assistant for CrediTrust. Your task is to answer questions about customer complaints. Use the following retrieved complaint excerpts to formulate your answer. If the context doesn't contain the answer, state that you don't have enough information.

Context:
{context}

Question: {question}

Answer:"""

    answer = generate_text(prompt)
    return answer, retrieved

print("RAG functions defined.")

RAG functions defined.


In [4]:
def respond(question, k=5):
    if not question or question.strip() == "":
        return "Please enter a question.", ""

    answer, retrieved = rag_answer(question, k)

    sources_text = ""
    if retrieved:
        sources_text = "\n\n" + "="*50 + "\n"
        sources_text += "SOURCES:\n"
        sources_text += "="*50 + "\n"
        for i, r in enumerate(retrieved):
            sources_text += f"\n[{i+1}] Score: {r['score']:.4f}\n"
            sources_text += f"Product: {r['product']}\n"
            sources_text += f"Text: {r['text'][:300]}\n"

    return answer, sources_text

print("Response function defined.")

Response function defined.


In [5]:
print("="*60)
print("LAUNCHING GRADIO UI")
print("="*60)

with gr.Blocks(title="CrediTrust Complaint Analyzer", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # CrediTrust Complaint Analyzer
    ### Ask questions about customer complaints across financial products
    """)

    with gr.Row():
        with gr.Column(scale=2):
            question_input = gr.Textbox(
                label="Your Question",
                placeholder="e.g., Why are customers unhappy with credit cards?",
                lines=2
            )

            with gr.Row():
                k_slider = gr.Slider(
                    minimum=1,
                    maximum=10,
                    value=5,
                    step=1,
                    label="Number of sources to retrieve (k)"
                )
                submit_btn = gr.Button("Ask", variant="primary", scale=1)
                clear_btn = gr.Button("Clear", variant="secondary", scale=1)

        with gr.Column(scale=3):
            answer_output = gr.Textbox(
                label="Answer",
                lines=8,
                interactive=False
            )
            sources_output = gr.Textbox(
                label="Sources",
                lines=10,
                interactive=False
            )

    gr.Markdown("""
    ---
    **Tip:** The answer is generated from customer complaint data. Sources show the exact complaint excerpts used.
    """)

    submit_btn.click(
        fn=respond,
        inputs=[question_input, k_slider],
        outputs=[answer_output, sources_output]
    )

    clear_btn.click(
        fn=lambda: ("", ""),
        inputs=[],
        outputs=[question_input, answer_output, sources_output]
    )

demo.launch(share=True, debug=False)
print("UI launched. Click the public URL to access.")

LAUNCHING GRADIO UI
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://702a46ec11ebad5c8c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


UI launched. Click the public URL to access.


In [6]:
app_code = '''
import gradio as gr
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import warnings
warnings.filterwarnings('ignore')

device = 'cuda' if torch.cuda.is_available() else 'cpu'

index = faiss.read_index('vector_store/index.faiss')
with open('vector_store/chunks.pkl', 'rb') as f:
    chunked_data = pickle.load(f)

embed_model = SentenceTransformer('all-MiniLM-L6-v2', device=device)

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
if device == 'cuda':
    model = model.to('cuda')

def generate_text(prompt):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    if device == 'cuda':
        inputs = {k: v.to('cuda') for k, v in inputs.items()}
    outputs = model.generate(**inputs, max_new_tokens=150)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

def retrieve(query, k=5):
    query_emb = embed_model.encode([query], normalize_embeddings=True).astype(np.float32)
    distances, indices = index.search(query_emb, k)
    results = []
    for i, idx in enumerate(indices[0]):
        if idx < len(chunked_data):
            results.append({
                'score': float(distances[0][i]),
                'product': chunked_data[idx]['product_category'],
                'text': chunked_data[idx]['chunk_text']
            })
    return results

def rag_answer(question, k=5):
    retrieved = retrieve(question, k)
    if not retrieved:
        return "No relevant information found.", []

    context = "\n\n".join([f"Excerpt {i+1}: {r['text']}" for i, r in enumerate(retrieved)])

    prompt = f"""You are a financial analyst assistant for CrediTrust. Your task is to answer questions about customer complaints. Use the following retrieved complaint excerpts to formulate your answer. If the context doesn't contain the answer, state that you don't have enough information.

Context:
{context}

Question: {question}

Answer:"""

    answer = generate_text(prompt)
    return answer, retrieved

def respond(question, k=5):
    if not question or question.strip() == "":
        return "Please enter a question.", ""

    answer, retrieved = rag_answer(question, k)

    sources_text = ""
    if retrieved:
        sources_text = "\n\n" + "="*50 + "\n"
        sources_text += "SOURCES:\n"
        sources_text += "="*50 + "\n"
        for i, r in enumerate(retrieved):
            sources_text += f"\n[{i+1}] Score: {r['score']:.4f}\n"
            sources_text += f"Product: {r['product']}\n"
            sources_text += f"Text: {r['text'][:300]}\n"

    return answer, sources_text

with gr.Blocks(title="CrediTrust Complaint Analyzer", theme=gr.themes.Soft()) as demo:
    gr.Markdown("""
    # CrediTrust Complaint Analyzer
    ### Ask questions about customer complaints across financial products
    """)

    with gr.Row():
        with gr.Column(scale=2):
            question_input = gr.Textbox(
                label="Your Question",
                placeholder="e.g., Why are customers unhappy with credit cards?",
                lines=2
            )

            with gr.Row():
                k_slider = gr.Slider(
                    minimum=1,
                    maximum=10,
                    value=5,
                    step=1,
                    label="Number of sources to retrieve (k)"
                )
                submit_btn = gr.Button("Ask", variant="primary", scale=1)
                clear_btn = gr.Button("Clear", variant="secondary", scale=1)

        with gr.Column(scale=3):
            answer_output = gr.Textbox(
                label="Answer",
                lines=8,
                interactive=False
            )
            sources_output = gr.Textbox(
                label="Sources",
                lines=10,
                interactive=False
            )

    gr.Markdown("""
    ---
    **Tip:** The answer is generated from customer complaint data. Sources show the exact complaint excerpts used.
    """)

    submit_btn.click(
        fn=respond,
        inputs=[question_input, k_slider],
        outputs=[answer_output, sources_output]
    )

    clear_btn.click(
        fn=lambda: ("", "", ""),
        inputs=[],
        outputs=[question_input, answer_output, sources_output]
    )

demo.launch(share=True, debug=False)
'''

with open('/content/app.py', 'w') as f:
    f.write(app_code)

print("app.py created in /content/")

app.py created in /content/


In [7]:
from google.colab import files
files.download('/content/app.py')
print("app.py downloaded to your computer.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

app.py downloaded to your computer.
